In [1]:
import os
import pandas as pd
import time
import json
import pytesseract
import re
import matplotlib.pyplot as plt
import numpy as np
import cv2
import shutil
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract  


<h1>Créé un dossier contenant les informations texte sur les images</h1>

In [7]:
def traiter_image(image_path, zone_texte, contraste=4):
    img = Image.open(image_path)
    
    img_crop = img.crop(zone_texte)

    img_crop_gray = img_crop.convert('L')

    enhancer = ImageEnhance.Contrast(img_crop_gray)
    img_crop_contraste = enhancer.enhance(contraste)

    texte = pytesseract.image_to_string(img_crop_contraste, lang='fra+eng')

    # # Afficher les images à chaque étape
    # plt.figure(figsize=(10, 6))

    # plt.subplot(1, 3, 1)
    # plt.title('Image Originale Recadrée')
    # plt.imshow(img_crop)
    # plt.axis('off')

    # plt.subplot(1, 3, 2)
    # plt.title('Niveaux de Gris')
    # plt.imshow(img_crop_gray, cmap='gray')
    # plt.axis('off')

    # plt.subplot(1, 3, 3)
    # plt.title('Contraste Amélioré')
    # plt.imshow(img_crop_contraste, cmap='gray')
    # plt.axis('off')

    # plt.tight_layout()
    # plt.show()

    return texte

def extraire_informations(texte):
    informations = {
        "drive_number": None,
        "gps": {
            "latitude": None,
            "longitude": None,
            "altitude": None
        },
        "position_km": None,
        "date": None,
        "time": None,
        "gps_quality": None,
        "width": None,
        "height": None,
        "speed_kmh": None,
        "additional_info": {
            "gebied": None,
            "geo_code": None,
            "turnout_number": None,
            "direction": None,
            "emplacement": None,
            "route": None
        }
    }
    
    try:
        # Drive No.
        drive_match = re.search(r'Drive No\.\s*(\d+)', texte)
        if drive_match:
            informations["drive_number"] = drive_match.group(1)

        # GPS Latitude
        latitude_match = re.search(r'GPS Latitude\s*([-\d.,]+)', texte)
        if latitude_match:
            informations["gps"]["latitude"] = float(latitude_match.group(1).replace(',', '.'))

        # Position
        position_match = re.search(r'Position\s*(.*?)\s*GPS', texte, re.DOTALL)
        if position_match:
            position_value = position_match.group(1)  # Récupère tout entre "Position" et "GPS"
            
            # Remplace les tirets longs par un simple tiret et supprime les espaces
            position_value = position_value.replace('—-', '-').replace('-', '-').replace('—','-').replace('kn','km').strip()

            print(position_value)  # Pour vérifier la valeur extraite
                # Vérifie si la valeur restante peut être convertie en float
            if position_value:
                try:
                    informations["position_km"] = float(position_value.replace(',', '.'))  
                except ValueError:
                    informations["position_km"] = None  
            else:
                informations["position_km"] = None  
        else:
            informations["position_km"] = None  



        # GPS Longitude
        longitude_match = re.search(r'GPS Longitude\s*([-\d.,\s]+)', texte)
        if longitude_match:
            longitude_value = longitude_match.group(1).replace(',', '.').strip().replace(' ', '')
            informations["gps"]["longitude"] = float(longitude_value)

        # Altitude
        altitude_match = re.search(r'GPS Altitude\s*([-\d.,]+)', texte)
        if altitude_match:
            informations["gps"]["altitude"] = float(altitude_match.group(1).replace(',', '.'))

        # Date
        date_match = re.search(r'Date\s*([\d/]+)', texte)
        if date_match:
            informations["date"] = date_match.group(1)

        # Time
        time_match = re.search(r'Tine\s*([\d:.,]+)', texte)
        if time_match:
            informations["time"] = time_match.group(1)

        # GPS Quality
        quality_match = re.search(r'GPS Quality\s*([\d.]+)', texte)
        if quality_match:
            informations["gps_quality"] = float(quality_match.group(1))

        # Width
        width_match = re.search(r'Width\s*([^\n]*)', texte)
        if width_match:
            width_value = width_match.group(1).strip()
            informations["width"] = "0mm" if "Onn" or "O nn" or "O mm" in width_value else width_value

        # Height
        height_match = re.search(r'Heigth\s*([^\n]*)', texte)
        if height_match:
            height_value = height_match.group(1).strip()
            informations["height"] = "0mm" if "Onn" in height_value or "O nn" or "O mm" in height_value else height_value

        # Speed
        speed_match = re.search(r'Speed\s*([-\d.,]+)\s*k(?:mh|wh)', texte)
        if speed_match:
            informations["speed_kmh"] = float(speed_match.group(1).replace(',', '.'))

        # Additional Info
        additional_info_matches = {
            "gebied": re.search(r'Gebied\s*([^\n]*)', texte),
            "geo_code": re.search(r'Geo Code[:\s]*([^\n]*)', texte),
            "turnout_number": re.search(r'Turnout #:\s*([^\n]*)', texte),
            "direction": re.search(r'Direction:\s*([^\n]*)', texte),
            "emplacement": re.search(r'Enplacenent\s*([^\n]*)', texte),
            "route": re.search(r'Route:\s*([^\n]*)', texte),
        }

        for key, match in additional_info_matches.items():
            if match:
                informations["additional_info"][key] = match.group(1).strip()
            else:
                informations["additional_info"][key] = "Unknown"  
        
    except ValueError as e:
        print(f"Erreur lors de l'extraction des informations : {e}")
    except Exception as e:
        print(f"Erreur générale : {e}")
    
    return informations

def parcourir_repertoire(repertoire, zone_texte, repertoire_sortie):
    if not os.path.exists(repertoire_sortie):
        os.makedirs(repertoire_sortie)
    
    for fichier in os.listdir(repertoire):
        if fichier.lower().endswith(('.jpg', '.jpeg', '.png')):
            chemin_image = os.path.join(repertoire, fichier)
            

            texte = traiter_image(chemin_image, zone_texte)
            

            resultat = {
                'nom_image': fichier,
                'texte_reconnu': texte
            }


            informations = extraire_informations(texte)
            resultat['informations'] = informations
            

            nom_fichier_json = os.path.splitext(fichier)[0] + '.json'
            chemin_json = os.path.join(repertoire_sortie, nom_fichier_json)
            with open(chemin_json, 'w', encoding='utf-8') as json_file:
                json.dump(resultat, json_file, ensure_ascii=False, indent=4)

In [ ]:
repertoire_images = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/9820_up"
repertoire_sortie = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc"
zone_texte = (0, 0, 420, 180)

parcourir_repertoire(repertoire_images, zone_texte, repertoire_sortie)

print("Traitement terminé.")

-0,396km
-0,393km
-0,390km
-0.387km
-0.384km
-0.381km
1,002km
Erreur lors de l'extraction des informations : could not convert string to float: '7..7317600'
-0.378km
-0.375km
-0.375km
-0,369km
-0.366km
-0.366km
-0.360km
-0.357km
-0.357km
-0.351km
1,005km
-0,348km
-0.345km
-0.342km
-0,339km
-0,336km
-0,333km
-0.330km
-0.327km
-0.324km
-0.321km
1,008km
Erreur lors de l'extraction des informations : could not convert string to float: '7..7317600'
-0,318km
-0.315km
-0.315km
-0,309km
-0,306km
-0,303km
-0,300km
-0.297km
-0,294km
-0,291km
1,011km
-0,288km
-0,288km
-0.282km
-0.279km
-0.279km
-0.273km
-0.273km
-0.267km
-0.267km
-0.261km
1,014km
-0.258km
-0.255km
-0.252km
-0.249km
-0.246km
-0.246km
-0.240km
-0.237km
-0.234km
-0.231km
1,017km
-0.228km
-0.225km
-0.222km
-0.219km
-0.216km
-0.216km
-0.210km
-0.207km
-0,204km
-0,201km
0,102km ©
1,020km
-0,198km
-0,195km
-0,192km
-0,189km
-0,186km
-0,183km
-0,180km
-0,180km
-0.174km
-0.174km
1,023km
Erreur lors de l'extraction des informations : could

<h1>Créé un json unique regroupant les informations temporelles</h1>

In [20]:
import os
import json
import re

# Chemin du dossier contenant les fichiers JSON
dossier_json = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc"
fichier_sortie = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json/timestamps_extraits.json"

# Regex pour capturer le format HH:MM:SS,XX
time_pattern = re.compile(r"\b\d{2}:\d{2}:\d{2},\d{2}\b")

timestamps = []

# Parcours des fichiers JSON dans le dossier
for fichier in os.listdir(dossier_json):
    if fichier.endswith(".json"):  # Vérifie si c'est un fichier JSON
        chemin_fichier = os.path.join(dossier_json, fichier)

        with open(chemin_fichier, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)  # Charge le contenu JSON
                texte = data.get("texte_reconnu", "")

                # Recherche du timestamp dans le texte
                match = time_pattern.search(texte)
                if match:
                    timestamps.append({"fichier": fichier, "timestamp": match.group()})

            except json.JSONDecodeError:
                print(f"Erreur lors de la lecture de {fichier}")

# Sauvegarde dans un fichier JSON
with open(fichier_sortie, "w", encoding="utf-8") as f_out:
    json.dump(timestamps, f_out, indent=4, ensure_ascii=False)

print(f"Timestamps extraits et sauvegardés dans {fichier_sortie}")


Timestamps extraits et sauvegardés dans /Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json/timestamps_extraits.json


In [54]:
dossier_json = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc"
fichier_sortie = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json/timestamps_extraits_debug.json"

time_pattern = re.compile(r"\b\d{2}:\d{2}:\d{2}[.,]\d+\b")

timestamps = []

for fichier in os.listdir(dossier_json):
    if fichier.endswith(".json"):
        chemin_fichier = os.path.join(dossier_json, fichier)

        if os.path.getsize(chemin_fichier) == 0:
            print(f"⚠️  Fichier vide ignoré : {fichier}")
            continue  

        with open(chemin_fichier, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
                texte = data.get("texte_reconnu", "")

                if "texte_reconnu" not in data:
                    print(f"⚠️  Clé 'texte_reconnu' absente dans {fichier}")

                match = time_pattern.search(texte)
                if match:
                    timestamps.append({"fichier": fichier, "timestamp": match.group()})
                else:
                    print(f"⏳ Aucun timestamp trouvé dans {fichier}")

            except json.JSONDecodeError:
                print(f"❌ Erreur JSON dans {fichier}")

with open(fichier_sortie, "w", encoding="utf-8") as f_out:
    json.dump(timestamps, f_out, indent=4, ensure_ascii=False)

print(f"✅ {len(timestamps)} fichiers traités et sauvegardés dans {fichier_sortie}")


⏳ Aucun timestamp trouvé dans 240902_9820_RC_11922000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_16269000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42792000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42795000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42798000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42801000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42804000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_42807000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47139000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47142000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47145000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47148000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47151000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47154000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47157000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_47160000.json
⏳ Aucun timestamp trouvé dans 240902_9820_RC_48357000.js

<h1>Donne la liste des fichiers correspondants au timestamp recherché</h1>

In [64]:
import json
from datetime import datetime, timedelta

def trouver_extremes_timestamps(fichier_json):
    """Extrait tous les timestamps et retourne le plus tôt et le plus tard."""
    try:
        with open(fichier_json, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        timestamps = []
        
        for item in data:
            timestamp_str = item.get("timestamp")
            if timestamp_str:
                try:
                    # Remplacement du point par une virgule pour uniformiser
                    timestamp_str = timestamp_str.replace('.', ',')
                    timestamp_dt = datetime.strptime(timestamp_str, "%H:%M:%S,%f")
                    
                    # Ajouter un jour fictif pour gérer les timestamps entre minuit et 1h
                    if timestamp_dt.hour < 1:  # Entre minuit et 1h
                        timestamp_dt = timestamp_dt.replace(year=2024, month=1, day=2)  # Jour suivant
                    else:  # Entre 1h et 23h59
                        timestamp_dt = timestamp_dt.replace(year=2024, month=1, day=1)  # Jour actuel
                    
                    timestamps.append(timestamp_dt)
                except ValueError:
                    print(f"Format invalide ignoré: {timestamp_str}")
        
        if timestamps:
            return min(timestamps), max(timestamps)
        else:
            return None, None

    except FileNotFoundError:
        print(f"Fichier non trouvé: {fichier_json}")
        return None, None
    except json.JSONDecodeError:
        print("Erreur de lecture du JSON.")
        return None, None

# Exemple d'utilisation
fichier_timestamps = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json/timestamps_extraits_debug.json"
min_timestamp, max_timestamp = trouver_extremes_timestamps(fichier_timestamps)

if min_timestamp and max_timestamp:
    # Afficher uniquement l'heure, les minutes, les secondes et les millisecondes
    print(f"Premier timestamp : {min_timestamp.strftime('%H:%M:%S,%f')[:-3]}")
    print(f"Dernier timestamp : {max_timestamp.strftime('%H:%M:%S,%f')[:-3]}")
else:
    print("Aucun timestamp valide trouvé.")

Format invalide ignoré: 21:81:41,87
Format invalide ignoré: 21:81:50,43
Format invalide ignoré: 21:81:52,41
Format invalide ignoré: 21:81:54,30
Format invalide ignoré: 21:84:06,40
Format invalide ignoré: 21:84:07,83
Format invalide ignoré: 21:84:09,11
Format invalide ignoré: 21:84:10,31
Format invalide ignoré: 21:84:11,45
Format invalide ignoré: 21:84:12,585
Format invalide ignoré: 21:84:13,62
Format invalide ignoré: 21:84:14,67
Format invalide ignoré: 21:84:15,67
Format invalide ignoré: 21:84:16,59
Format invalide ignoré: 21:84:17,45
Format invalide ignoré: 21:84:18,26
Format invalide ignoré: 21:84:19,03
Format invalide ignoré: 21:84:20,48
Format invalide ignoré: 21:84:21,18
Format invalide ignoré: 21:84:21,86
Format invalide ignoré: 21:84:22,581
Format invalide ignoré: 21:84:23,14
Format invalide ignoré: 21:84:24,31
Format invalide ignoré: 21:84:24,87
Format invalide ignoré: 21:84:25,42
Format invalide ignoré: 21:84:27,03
Format invalide ignoré: 21:84:27,03
Format invalide ignoré: 21

In [13]:
fichier_timestamps = fichier_sortie

def trouver_fichiers_par_timestamp(timestamp_recherche):
    """Recherche les fichiers contenant un timestamp donné."""
    try:
        with open(fichier_timestamps, "r", encoding="utf-8") as f:
            timestamps_data = json.load(f)  # Charger le JSON
        
        # Filtrer les fichiers correspondant au timestamp recherché
        fichiers_correspondants = [item["fichier"] for item in timestamps_data if item["timestamp"] == timestamp_recherche]
        
        return fichiers_correspondants
    
    except FileNotFoundError:
        print("Le fichier des timestamps n'existe pas.")
        return []
    except json.JSONDecodeError:
        print("Erreur de lecture du fichier JSON.")
        return []



In [44]:
import json
from datetime import datetime, timedelta

fichier_timestamps = fichier_sortie

def trouver_fichiers_par_timestamp(timestamp_recherche):
    """Recherche les fichiers contenant un timestamp dans l'intervalle [timestamp_recherche - 1s; timestamp_recherche + 1s]."""
    try:
        # Convertir le timestamp_recherche en objet datetime
        timestamp_recherche = datetime.strptime(timestamp_recherche, "%Y-%m-%d %H:%M:%S.%f")
        
        # Définir l'intervalle de recherche
        intervalle_min = timestamp_recherche - timedelta(seconds=1)
        intervalle_max = timestamp_recherche + timedelta(seconds=1)
        
        # Charger le fichier JSON
        with open(fichier_timestamps, "r", encoding="utf-8") as f:
            timestamps_data = json.load(f)
        
        # Liste pour stocker les fichiers correspondants
        fichiers_correspondants = []
        
        # Parcourir les données du JSON
        for item in timestamps_data:
            # Extraire le timestamp du JSON (format "20:32:57,19")
            timestamp_str = item["timestamp"]
            
            # Convertir le timestamp du JSON en objet datetime (en utilisant la date de timestamp_recherche)
            try:
                timestamp_fichier = datetime.strptime(
                    f"{timestamp_recherche.date()} {timestamp_str.replace(',', '.')}", 
                    "%Y-%m-%d %H:%M:%S.%f"
                )
            except ValueError:
                continue  # Ignorer les timestamps mal formatés
            
            # Vérifier si le timestamp est dans l'intervalle
            if intervalle_min <= timestamp_fichier <= intervalle_max:
                # Remplacer .json par .jpg dans le nom du fichier
                fichier_jpg = item["fichier"].replace(".json", ".jpg")
                fichiers_correspondants.append(fichier_jpg)
        
        return fichiers_correspondants
    
    except FileNotFoundError:
        print("Le fichier des timestamps n'existe pas.")
        return []
    except json.JSONDecodeError:
        print("Erreur de lecture du fichier JSON.")
        return []

In [42]:
timestamp_recherche = "2024-09-09 00:06:43.378350"
fichiers_correspondants = trouver_fichiers_par_timestamp(timestamp_recherche)
print("Fichiers correspondants :", fichiers_correspondants)

Fichiers correspondants : ['240902_9820_RC_76845000.jpg', '240902_9820_RC_76848000.jpg', '240902_9820_RC_76851000.jpg', '240902_9820_RC_76854000.jpg', '240902_9820_RC_76857000.jpg', '240902_9820_RC_76860000.jpg', '240902_9820_RC_76866000.jpg', '240902_9820_RC_76869000.jpg', '240902_9820_RC_76872000.jpg']


In [30]:
# Exemple d'utilisation
timestamp_a_rechercher = "2024-09-09 15:47:10.409350"
resultats = trouver_fichiers_par_timestamp(timestamp_a_rechercher, fichier_timestamps="/Volumes/Seagate Drive/pdi/nouvelle_tournee/sortie_rc_json/timestamps_extraits.json")

if resultats:
    print(f"Le timestamp {timestamp_a_rechercher} est présent dans les fichiers suivants :")
    for fichier in resultats:
        print(f"- {fichier}")
else:
    print(f"Aucun fichier ne contient le timestamp {timestamp_a_rechercher}.")

Erreur de format des timestamps.
Aucun fichier ne contient le timestamp 2024-09-09 15:47:10.409350.


In [17]:
# Chemin du dossier contenant les images
dossier_images = "/Volumes/Seagate Drive/pdi/nouvelle_tournee/test"

# Nom de l'image à afficher
nom_image = "240902_9820_RC_6000.jpg"

# Charger et afficher l'image
chemin_image = os.path.join(dossier_images, nom_image)
image = Image.open(chemin_image)
image.show()
